In [443]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader

In [444]:
air_quality_dataset = pd.read_csv('/content/week3_air_quality_hourly_20260305_145306.csv')

In [445]:
x = air_quality_dataset.drop(columns=['city', 'state', 'zip', 'time', 'us_aqi'])
y = air_quality_dataset['us_aqi']

In [446]:
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, shuffle=True)

In [447]:
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42, shuffle=True)

In [448]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_val = scaler.transform(X_val)

In [449]:
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val.to_numpy(), dtype=torch.float32).view(-1,1)
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.float32).view(-1,1)

In [450]:
X_train.shape

(13409, 10)

In [451]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

In [452]:
dataIter = iter(train_loader)
data, aqi = next(dataIter)
print(torch.min(data), torch.max(data))

tensor(-2.0238) tensor(5.4375)


In [453]:
class autoencoder(nn.Module):
  def __init__(self, input_dims):
    super().__init__()

    self.encoder = nn.Sequential(
        nn.Linear(input_dims, 8),
        nn.ReLU(),
        nn.Linear(8,4)
    )
    self.decoder = nn.Sequential(
        nn.Linear(4,8),
        nn.ReLU(),
        nn.Linear(8, input_dims)
    )
  def forward(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded

In [454]:
model = autoencoder(X_train_tensor.shape[1])
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [455]:
epochs = 20
for e in range(epochs):
  sum_batch = 0
  model.train()

  for X_batch, _ in train_loader:
    reconstruction = model(X_batch)

    loss = criterion(reconstruction, X_batch)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    sum_batch = sum_batch + loss.item()
  model.eval()

  with torch.no_grad():
    val_reconstruction = model(X_val_tensor)
    val_loss = criterion(val_reconstruction, X_val_tensor)
  print(f'Epoch {e+1}: Loss: train={(sum_batch / len(train_loader)):.4f}, val={val_loss.item():.4f}')

Epoch 1: Loss: train=0.9898, val=0.8975
Epoch 2: Loss: train=0.8439, val=0.7730
Epoch 3: Loss: train=0.7400, val=0.6889
Epoch 4: Loss: train=0.6593, val=0.5955
Epoch 5: Loss: train=0.5323, val=0.4208
Epoch 6: Loss: train=0.3653, val=0.2981
Epoch 7: Loss: train=0.2749, val=0.2403
Epoch 8: Loss: train=0.2328, val=0.2147
Epoch 9: Loss: train=0.2135, val=0.2006
Epoch 10: Loss: train=0.2007, val=0.1905
Epoch 11: Loss: train=0.1909, val=0.1819
Epoch 12: Loss: train=0.1820, val=0.1739
Epoch 13: Loss: train=0.1741, val=0.1656
Epoch 14: Loss: train=0.1651, val=0.1554
Epoch 15: Loss: train=0.1544, val=0.1444
Epoch 16: Loss: train=0.1447, val=0.1359
Epoch 17: Loss: train=0.1366, val=0.1293
Epoch 18: Loss: train=0.1296, val=0.1234
Epoch 19: Loss: train=0.1235, val=0.1184
Epoch 20: Loss: train=0.1181, val=0.1139


In [456]:
model.eval()

with torch.no_grad():
  test_reconstruction = model(X_test_tensor)
  test_loss = criterion(test_reconstruction, X_test_tensor)
  print(f'Test Loss={test_loss.item():.4f}')

Test Loss=0.1154
